# Gradient boosting / XGBoost (eXtreme Gradient Boosting)

_Classical ML_

**Build the model in stages: each new tree fixes the last one's mistakes.**

Random forests build many trees in parallel and vote. Gradient boosting builds them one at a time, in sequence.

     Each new tree is trained to fix what the current model still gets wrong — its residual errors.

---

This notebook is a practice scaffold for the **Gradient boosting / XGBoost (eXtreme Gradient Boosting)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a regression dataset and split it

Gradient boosting is easiest to watch on a regression task, where each tree predicts a number. We synthesise 400 examples with 6 features and a little noise, then hold out a quarter of them as a test set. The model never sees the test rows during training, so the test error is an honest estimate of how well boosting generalises.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# 400 examples, 6 features, mild Gaussian noise on the target.
X, y = make_regression(n_samples=400, n_features=6, noise=10.0, random_state=0)

# Hold out 25% as an unseen test set.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Fit the boosted ensemble

`GradientBoostingRegressor` builds 200 shallow trees (depth 3) one after another. Each tree is fit to the current model's residual errors, and `learning_rate=0.1` shrinks every tree's contribution so the ensemble improves in small, steady steps rather than overfitting in one leap.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                               max_depth=3, random_state=0)
gb.fit(Xtr, ytr)

print("n_estimators:", gb.n_estimators_)

### Step 3 — Score on the test set

We predict the held-out rows and report two standard regression scores: R² (how much of the target's variance the model explains, closer to 1 is better) and mean squared error (average squared mistake, lower is better).

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = gb.predict(Xte)

test_r2 = r2_score(yte, pred)
test_mse = mean_squared_error(yte, pred)
print("test R^2:", round(test_r2, 3))
print("test MSE:", round(test_mse, 2))

### Step 4 — Watch the error shrink as trees accumulate

`staged_predict` replays the ensemble's prediction after each successive tree is added. Checking the test MSE after 1, 50, and 200 trees shows the core idea of boosting: error falls sharply at first and then flattens as later trees have less left to fix.

In [ ]:
# staged_predict yields the prediction after 1 tree, then 2, ... up to 200.
for i, yp in enumerate(gb.staged_predict(Xte), start=1):
    if i in (1, 50, 200):
        stage_mse = mean_squared_error(yte, yp)
        print("after %3d trees  test MSE = %.2f" % (i, stage_mse))

## Visualize the data & results

_On the Breast Cancer data, does held-out error keep shrinking as boosting rounds accumulate?_

### Step 1 — Train a boosting classifier on Breast Cancer

Now switch to a real classification task. We split the breast-cancer data (stratified, so both classes keep their proportions) and fit a `GradientBoostingClassifier` with the same 200-tree, depth-3 recipe.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

bc = load_breast_cancer()

# Stratify keeps the malignant/benign ratio identical in train and test.
Xtr, Xte, ytr, yte = train_test_split(
    bc.data, bc.target, test_size=0.25, random_state=0,
    stratify=bc.target)

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                max_depth=3, random_state=0).fit(Xtr, ytr)

### Step 2 — Plot the test log-loss per boosting round

`staged_predict_proba` gives the class probabilities after each round, and `log_loss` measures how well those probabilities match the true labels (lower is better). Plotting it against the number of trees shows whether held-out error keeps shrinking — and where it stops paying off.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss

# One log-loss value per boosting round on the held-out test set.
rounds = np.arange(1, gb.n_estimators_ + 1)
loss = np.array([log_loss(yte, p)
                 for p in gb.staged_predict_proba(Xte)])

plt.plot(rounds, loss, c="#4ea1ff")
plt.title("Gradient boosting on Breast Cancer: test log-loss per round")
plt.xlabel("number of trees")
plt.ylabel("test log-loss")
plt.show()